In [0]:
#imports and setup

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog novacart_casestudy")
spark.sql("create schema if not exists silver_schema")

silver_run_id = str(uuid.uuid4())
print("current silver Run id :", silver_run_id)

In [0]:
#create ingestion control table
spark.sql("""
          create table if not exists novacart_casestudy.silver_schema.processing_control(
              
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_bronze_ingested_at timestamp,
              rows_merged bigint,
              run_status string,
              silver_run_id string,
              updated_at timestamp
          )
          using delta
          """)

In [0]:
def upsert_to_silver(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark,target_table)
        (dt.alias("target")
        .merge(df_source.alias("source"),f"target.{join_key} = source.{join_key}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_bronze_ingested_at(entity_name:str):
    ctrl = (
        spark.table("novacart_casestudy.silver_schema.processing_control")
                .filter(
                    (F.col("layer") == "silver") &
                    (F.col("entity_name") == entity_name) &
                    (F.col("run_status") == 'SUCCESS')
                )
                .orderBy (F.col("updated_at").desc())
                .limit(1)
    )

    rows = ctrl.collect()
    if not rows:
        return None,None

    return rows[0]["last_processed_bronze_ingested_at"],rows[0]["last_processed_bronze_run_id"]



In [0]:
from datetime import datetime, UTC

def upsert_silver_control(
    entity_name,
    last_processed_bronze_run_id,
    last_processed_bronze_ingested_at,
    rows_merged,
    silver_run_id
):
    ctrl_df = spark.createDataFrame(
        [(
            "silver",
            entity_name,
            last_processed_bronze_run_id,
            last_processed_bronze_ingested_at,
            int(rows_merged),
            "SUCCESS",
            silver_run_id,
            datetime.now(UTC).replace(tzinfo=None)
        )],
        schema="""
            layer string,
            entity_name string,
            last_processed_bronze_run_id string,
            last_processed_bronze_ingested_at timestamp,
            rows_merged bigint,
            run_status string,
            silver_run_id string,
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(
        spark,
        "novacart_casestudy.silver_schema.processing_control"
    )

    (
        dt.alias("t")
        .merge(
            ctrl_df.alias("s"),
            "t.layer = s.layer AND t.entity_name = s.entity_name"
        )
        .whenMatchedUpdate(set={
            "last_processed_bronze_run_id": "s.last_processed_bronze_run_id",
            "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
            "rows_merged": "s.rows_merged",
            "run_status": "s.run_status",
            "silver_run_id": "s.silver_run_id",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
def get_incremental_bronze(bronze_table,entity_name):
    last_ingested_at,last_run_id = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df = spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df,last_ingested_at,last_run_id
    return bronze_df.filter(F.col("bronze_ingested_at") > F.lit(last_ingested_at)),last_ingested_at,last_run_id

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime, UTC

# create processing control table
spark.sql("""
CREATE TABLE IF NOT EXISTS novacart_casestudy.silver_schema.processing_control (
    layer STRING,
    entity_name STRING,
    last_processed_bronze_run_id STRING,
    last_processed_bronze_ingested_at TIMESTAMP,
    rows_merged BIGINT,
    run_status STRING,
    silver_run_id STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

def upsert_to_silver(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (
            dt.alias("target")
            .merge(
                df_source.alias("source"),
                f"target.{join_key} = source.{join_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        df_source.write.format("delta").mode("overwrite").saveAsTable(target_table)


def get_last_processed_bronze_checkpoint(entity_name: str):
    rows = (
        spark.table("novacart_casestudy.silver_schema.processing_control")
        .filter(
            (F.col("layer") == "silver") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "SUCCESS")
        )
        .select(
            "last_processed_bronze_ingested_at",
            "last_processed_bronze_run_id"
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
        .collect()
    )

    if not rows:
        return None, None

    return (
        rows[0]["last_processed_bronze_ingested_at"],
        rows[0]["last_processed_bronze_run_id"]
    )


def upsert_silver_control(
    entity_name,
    last_processed_bronze_run_id,
    last_processed_bronze_ingested_at,
    rows_merged,
    silver_run_id
):
    ctrl_df = spark.createDataFrame(
        [(
            "silver",
            entity_name,
            last_processed_bronze_run_id,
            last_processed_bronze_ingested_at,
            int(rows_merged),
            "SUCCESS",
            silver_run_id,
            datetime.now(UTC).replace(tzinfo=None)
        )],
        schema="""
            layer string,
            entity_name string,
            last_processed_bronze_run_id string,
            last_processed_bronze_ingested_at timestamp,
            rows_merged bigint,
            run_status string,
            silver_run_id string,
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(
        spark,
        "novacart_casestudy.silver_schema.processing_control"
    )

    (
        dt.alias("t")
        .merge(
            ctrl_df.alias("s"),
            "t.layer = s.layer AND t.entity_name = s.entity_name"
        )
        .whenMatchedUpdate(set={
            "last_processed_bronze_run_id": "s.last_processed_bronze_run_id",
            "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
            "rows_merged": "s.rows_merged",
            "run_status": "s.run_status",
            "silver_run_id": "s.silver_run_id",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )


def get_incremental_bronze(bronze_table, entity_name):
    last_ingested_at, last_run_id = get_last_processed_bronze_checkpoint(entity_name)
    bronze_df = spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df, last_ingested_at, last_run_id

    incremental_df = bronze_df.filter(
        (F.col("bronze_ingested_at") > F.lit(last_ingested_at)) |
        (
            (F.col("bronze_ingested_at") == F.lit(last_ingested_at)) &
            (F.col("bronze_run_id") > F.lit(last_run_id))
        )
    )

    return incremental_df, last_ingested_at, last_run_id

### orders incremental processing

In [0]:
df_raw = spark.sql("select * from novacart_casestudy.bronze_schema.orders_raw")
display(df_raw)

In [0]:
# step 4 - orders incremental processing

orders_inc, last_orders_ingested_at, last_orders_run_id = get_incremental_bronze(
    "novacart_casestudy.bronze_schema.orders_raw",
    "orders"
)

orders_inc_count = orders_inc.count()
print(f"orders rows_to_process_silver = {orders_inc_count}")

if orders_inc_count > 0:
    order_window = Window.partitionBy("order_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    orders_cleaned = (
        orders_inc
        .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
        .withColumn(
            "order_status",
            F.when(F.col("order_status") == "", F.lit(None)).otherwise(F.col("order_status"))
        )
        .withColumn("order_amount", F.trim(F.col("order_amount")))
        .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r"\$", ""))
        .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r",", ""))
        .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r"\s+", ""))
        .withColumn(
            "order_amount",
            F.when(
                F.col("order_amount").isin("N/A", "NULL", "?", ""),
                F.lit(None)
            ).otherwise(F.col("order_amount"))
        )
        .withColumn("order_amount", F.col("order_amount").cast("double"))
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("updated_at", F.to_timestamp("updated_at"))
        .withColumn("row_rank", F.row_number().over(order_window))
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    upsert_to_silver(
        orders_cleaned,
        "novacart_casestudy.silver_schema.orders_cleaned",
        "order_id"
    )

    orders_validated = (
        orders_cleaned
        .withColumn(
            "to_be_verified_orders_team",
            F.when(F.col("customer_id").isNull(), "verify_customer_id")
             .when(F.col("product_id").isNull(), "verify_product_id")
             .when(
                 F.col("order_status").isNull() | (F.trim(F.col("order_status")) == ""),
                 "verify_order_status"
             )
             .when(
                 F.col("order_amount").isNull() | (F.col("order_amount") <= 0),
                 "verify_order_amount"
             )
             .otherwise("No Issues")
        )
        .withColumn(
            "check_order_amount",
            F.when(
                F.col("order_amount").isNull() | (F.col("order_amount") <= 0),
                F.lit(True)
            ).otherwise(F.lit(False))
        )
        .withColumn("order_date", F.to_date("created_at"))
        .withColumn("order_year", F.year("order_date"))
        .withColumn("order_month", F.month("order_date"))
        .withColumn("order_day", F.dayofmonth("order_date"))
        .withColumn("order_dayofweek", F.dayofweek("order_date"))
    )

    orders_good = orders_validated.filter(
        (F.col("to_be_verified_orders_team") == "No Issues") &
        (F.col("check_order_amount") == F.lit(False))
    )

    orders_bad = (
        orders_validated
        .filter(
            (F.col("to_be_verified_orders_team") != "No Issues") |
            (F.col("check_order_amount") == F.lit(True))
        )
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    upsert_to_silver(
        orders_good,
        "novacart_casestudy.silver_schema.orders_transformed",
        "order_id"
    )

    orders_bad.write.format("delta").mode("append").saveAsTable(
        "novacart_casestudy.silver_schema.orders_quarantine"
    )

    mx_ingested = (
        orders_inc
        .agg(F.max("bronze_ingested_at").alias("mx"))
        .collect()[0]["mx"]
    )

    mx_run = (
        orders_inc
        .filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control(
        "orders",
        mx_run,
        mx_ingested,
        orders_good.count(),
        silver_run_id
    )

else:
    print("No new orders Bronze rows for silver.")
    upsert_silver_control(
        "orders",
        last_orders_run_id,
        last_orders_ingested_at,
        0,
        silver_run_id
    )

In [0]:
%sql
select * from novacart_casestudy.silver_schema.orders_cleaned;

In [0]:
%sql
select * from novacart_casestudy.silver_schema.orders_transformed;

In [0]:
%sql
select * from novacart_casestudy.silver_schema.orders_quarantine

###  Products incremental processing

In [0]:
%sql
select * from novacart_casestudy.bronze_schema.products_raw;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# read bronze products
products_bronze = spark.table("novacart_casestudy.bronze_schema.products_raw")

# get last processed bronze checkpoint from silver control
last_products_ingested_at, last_products_run_id = get_last_processed_bronze_checkpoint("products")
print(f"last processed bronze ingested at = {last_products_ingested_at}")
print(f"last processed bronze run id = {last_products_run_id}")

# re-read recent data with lookback buffer
if last_products_ingested_at is None:
    products_inc = products_bronze
else:
    products_inc = products_bronze.filter(
        F.col("bronze_ingested_at") >= F.lit(last_products_ingested_at) - F.expr("INTERVAL 3 DAYS")
    )

products_inc_count = products_inc.count()
print(f"products rows_to_process_in_silver = {products_inc_count}")

if products_inc_count > 0:
    product_window = Window.partitionBy("product_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    products_cleaned = (
        products_inc
        .withColumn("updated_at", F.to_timestamp("updated_at"))
        .withColumn("product_name", F.upper(F.col("product_name")))
        .withColumn("product_name", F.regexp_replace(F.col("product_name"), r"[-_]+", " "))
        .withColumn("product_name", F.regexp_replace(F.col("product_name"), r"\s+", " "))
        .withColumn("product_name", F.trim(F.col("product_name")))
        .withColumn(
            "product_name",
            F.when(
                F.col("product_name").isNull() | (F.col("product_name") == ""),
                F.lit(None)
            ).otherwise(F.col("product_name"))
        )
        .withColumn("category", F.upper(F.trim(F.col("category"))))
        .withColumn(
            "category",
            F.when(F.col("category").isin("", "NULL", "N/A", "?"), F.lit(None))
             .when(F.col("category").isin("LIFESTYLE", "ELECTRONICS", "FITNESS"), F.col("category"))
             .otherwise(F.col("category"))
        )
        .withColumn("price", F.trim(F.col("price")))
        .withColumn("price", F.regexp_replace(F.col("price"), r"\$", ""))
        .withColumn("price", F.regexp_replace(F.col("price"), ",", "."))
        .withColumn("price", F.regexp_replace(F.col("price"), r"\s+", ""))
        .withColumn("price", F.expr("try_cast(price as double)"))
        .withColumn("row_rank", F.row_number().over(product_window))
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    upsert_to_silver(
        products_cleaned,
        "novacart_casestudy.silver_schema.products_cleaned",
        "product_id"
    )

    products_validated = (
        products_cleaned
        .withColumn(
            "to_be_verified_by_product_team",
            F.when(F.col("product_name").isNull(), "verify_product_name")
             .when(F.col("price").isNull() | (F.col("price") <= 0), "verify_price")
             .when(F.col("category").isNull(), "verify_category")
             .when(~F.col("category").isin("LIFESTYLE", "ELECTRONICS", "FITNESS"), "verify_category")
             .otherwise("No Issues")
        )
        .withColumn(
            "check_product_price",
            F.when(
                F.col("price").isNull() | (F.col("price") <= 0),
                "invalid_price"
            ).otherwise("valid_price")
        )
    )

    products_good = products_validated.filter(
        (F.col("to_be_verified_by_product_team") == "No Issues") &
        (F.col("check_product_price") == "valid_price")
    )

    if "price_raw" in products_good.columns:
        products_good = products_good.drop("price_raw")

    products_bad = (
        products_validated
        .filter(
            (F.col("to_be_verified_by_product_team") != "No Issues") |
            (F.col("check_product_price") == "invalid_price")
        )
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    upsert_to_silver(
        products_good,
        "novacart_casestudy.silver_schema.products_transformed",
        "product_id"
    )

    products_bad.write.format("delta").mode("append").saveAsTable(
        "novacart_casestudy.silver_schema.products_quarantine"
    )

    mx_ingested = (
        products_inc
        .agg(F.max("bronze_ingested_at").alias("mx"))
        .collect()[0]["mx"]
    )

    mx_run = (
        products_inc
        .filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control(
        "products",
        mx_run,
        mx_ingested,
        products_good.count(),
        silver_run_id
    )

else:
    print("No products rows found for silver processing.")
    upsert_silver_control(
        "products",
        last_products_run_id,
        last_products_ingested_at,
        0,
        silver_run_id
    )

In [0]:
%sql
select * from novacart_casestudy.silver_schema.products_quarantine;

In [0]:
%sql
select * from novacart_casestudy.silver_schema.products_cleaned;

##Payments incremental processing
This cell process payments from Bronze to silver

it cleans:

payment_status,
paid_amount,
processed_at

Then it vaildates records,quarantines bad rows, and merge vaild rows into silver transformed payments table


In [0]:
payments_inc, last_payments_ingested_at, last_payments_run_id = get_incremental_bronze(
    "novacart_casestudy.bronze_schema.payments_raw",
    "payments"
)

print("Payments last processed Bronze ingested at =", last_payments_ingested_at)

payments_inc_count = payments_inc.count()
print(f"payments rows to process in silver = {payments_inc_count}")

if payments_inc_count > 0:
    payment_window = Window.partitionBy("payment_id").orderBy(
        F.col("processed_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    payments_cleaned = (
        payments_inc
        .withColumn("payment_status", F.upper(F.trim(F.col("payment_status"))))
        .withColumn(
            "payment_status",
            F.when(F.col("payment_status") == "", F.lit(None)).otherwise(F.col("payment_status"))
        )
        .withColumn("paid_amount", F.trim(F.col("paid_amount")))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), r"\$", ""))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), ",", "."))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), r"\s+", ""))
        .withColumn(
            "paid_amount",
            F.when(
                F.col("paid_amount").isin("N/A", "NULL", "?", ""),
                F.lit(None)
            ).otherwise(F.col("paid_amount"))
        )
        .withColumn("paid_amount", F.expr("try_cast(paid_amount as double)"))
        .withColumn("processed_at", F.to_timestamp("processed_at"))
        .withColumn("row_rank", F.row_number().over(payment_window))
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    upsert_to_silver(
        payments_cleaned,
        "novacart_casestudy.silver_schema.payments_cleaned",
        "payment_id"
    )

    payments_validated = (
        payments_cleaned
        .withColumn(
            "to_be_verified_by_payment_team",
            F.when(F.col("payment_id").isNull(), "verify_payment_id")
             .when(F.col("payment_status").isNull(), "verify_payment_status")
             .when(F.col("paid_amount").isNull() | (F.col("paid_amount") <= 0), "verify_paid_amount")
             .otherwise("No Issues")
        )
        .withColumn(
            "check_paid_amount",
            F.when(
                F.col("paid_amount").isNull() | (F.col("paid_amount") <= 0),
                F.lit(True)
            ).otherwise(F.lit(False))
        )
    )

    payments_good = payments_validated.filter(
        (F.col("to_be_verified_by_payment_team") == "No Issues") &
        (F.col("check_paid_amount") == F.lit(False))
    )

    payments_bad = (
        payments_validated
        .filter(
            (F.col("to_be_verified_by_payment_team") != "No Issues") |
            (F.col("check_paid_amount") == F.lit(True))
        )
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    upsert_to_silver(
        payments_good,
        "novacart_casestudy.silver_schema.payments_transformed",
        "payment_id"
    )

    payments_bad.write.format("delta").mode("append").saveAsTable(
        "novacart_casestudy.silver_schema.payments_quarantine"
    )

    mx_ingested = (
        payments_inc
        .agg(F.max("bronze_ingested_at").alias("mx"))
        .collect()[0]["mx"]
    )

    mx_run = (
        payments_inc
        .filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control(
        "payments",
        mx_run,
        mx_ingested,
        payments_good.count(),
        silver_run_id
    )

else:
    print("No new payments Bronze rows for silver.")
    upsert_silver_control(
        "payments",
        last_payments_run_id,
        last_payments_ingested_at,
        0,
        silver_run_id
    )

In [0]:
%sql
select * from novacart_casestudy.silver_schema.payments_cleaned;

In [0]:
%sql
select * from novacart_casestudy.silver_schema.payments_quarantine;

In [0]:
print(
    "Products transformed count:",
    spark.sql("SELECT COUNT(*) FROM novacart_casestudy.silver_schema.products_transformed").collect()[0][0]
)

print(
    "Orders transformed count:",
    spark.sql("SELECT COUNT(*) FROM novacart_casestudy.silver_schema.orders_transformed").collect()[0][0]
)

print(
    "Payments transformed count:",
    spark.sql("SELECT COUNT(*) FROM novacart_casestudy.silver_schema.payments_transformed").collect()[0][0]
)

display(
    spark.table("novacart_casestudy.silver_schema.processing_control")
    .orderBy("entity_name")
)